In [ ]:

import pandas as pd
from IPython.display import Markdown
from tabulate import tabulate

# Read your CSV and keep the relevant columns
df = pd.read_csv("rel_abs.csv").drop(columns=['file_path'], errors='ignore')
df['lr'] = df.get('lr', pd.NA).astype(str)  # keep lr as str if you need it later

# Keep the columns in a known order
df = df[['v. Loss', 'v. acc.', 'v. mIOU', 'lr', 'Weight', 'Reserve', 'Bands', 'Type']]

# 1) Melt metrics to long form
metrics_map = {'v. Loss':'loss','v. acc.':'acc','v. mIOU':'mIOU'}
long = df.melt(
    id_vars=['Bands','Type','Reserve'],
    value_vars=list(metrics_map.keys()),
    var_name='metric',
    value_name='value'
)
long['metric'] = long['metric'].map(metrics_map)

# 2) Pivot so columns are (Reserve, metric), rows are (Bands, Type)
wide = long.pivot_table(
    index=['Bands','Type'],
    columns=['Reserve','metric'],
    values='value',
    aggfunc='first'  # choose 'max' or 'mean' if you may have duplicates
)

# 3) Order columns by reserve then metric as in your sketch
reserve_order = ['BUS','ESK','HAM','KAU']  # add 'WAI' if needed
metric_order  = ['loss','acc','mIOU']

# Build the ordered column list that exists in the data
ordered_cols = []
for r in reserve_order:
    for m in metric_order:
        if (r, m) in wide.columns:
            ordered_cols.append((r, m))

# If there are other reserves present, append them after the main ones
for (r, m) in wide.columns:
    if (r, m) not in ordered_cols:
        ordered_cols.append((r, m))

# Reindex columns to desired order
wide = wide.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

# 4) Fill missing values (optional)
wide = wide.fillna('')

# 5) Flatten the MultiIndex columns to "Reserve metric" pairs for Markdown output
flat_cols = [f"{r} {m}" for (r, m) in wide.columns]
wide.columns = flat_cols

# 6) Reset index so Bands and Type are regular columns, round numeric-like cells
out = wide.reset_index()

# Round only numeric columns (keep blanks as '')
for c in out.columns:
    if c not in ['Bands','Type'] and pd.api.types.is_numeric_dtype(out[c]):
        out[c] = out[c].round(2)

# Print as a GitHub-flavored Markdown table
Markdown(tabulate(out, headers='keys', tablefmt='github', showindex=False))